# Band structures: DFT and Wannier-interpolated

Replaces the per-run `band_plot.ipynb`, `PTB_YIO_*.ipynb`, `pythtb_MnBi2Te4.ipynb` and the MnBi2Te4 `bands*.ipynb`.
Set the paths in the settings cell.

* the **DFT bands** come from a pw.x `calculation='bands'` output (`BANDS_OUT`); spin-polarised runs show both spins;
* each **Wannier run** in `WANNIER` is drawn over them, evaluated at exactly the same k-points (read from the
  `K_POINTS crystal_b` block of `BANDS_IN`).

Energies are relative to the SCF Fermi level (or highest occupied level).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "modules/axion.py").exists())
DATA = ROOT / "data"
sys.path.insert(0, str(ROOT))
from pythtb import W90
from modules.qe_read_file import read_bands_out, read_kpath, fermi_energy

In [ ]:
PREFIX = "SrTiO3"
# DFT bands: the pw.x bands output, its input (k-path and labels) and the SCF output (Fermi level; None: absolute energies)
BANDS_OUT = DATA / "SrTiO3/base/soc/output/181642bc889/bands/182493bc889/SrTiO3.bands.out"
BANDS_IN = DATA / "SrTiO3/base/soc/output/181642bc889/bands/182493bc889/SrTiO3.bands.in"
SCF_OUT = DATA / "SrTiO3/base/soc/output/181642bc889/SrTiO3.scf.out"
# Wannier runs drawn over the DFT bands, {label: folder}; {} for DFT only
WANNIER = {
    "set c (48 WF)": DATA / "SrTiO3/base/soc/output/181642bc889/trial_03_Sr_sp_Ti_pd_O_sp/185116bc889",
}
E_WINDOW = (-6.0, 4.0)          # eV relative to E_F
SHOW_HOPPINGS = True            # |H(R)| versus distance for each Wannier run

In [ ]:
k_cart, bands = read_bands_out(BANDS_OUT)
points, ticks = read_kpath(BANDS_IN)
if len(k_cart) == 2 * len(points):  # collinear spin-polarised: pw.x lists every k once per spin
    k_cart, bands = k_cart[: len(points)], np.vstack([bands[:, : len(points)], bands[:, len(points):]])
distance = np.r_[0, np.cumsum(np.linalg.norm(np.diff(k_cart, axis=0), axis=1))]
e_fermi = fermi_energy(SCF_OUT) if SCF_OUT else 0.0
print(f"DFT bands: {BANDS_OUT.relative_to(DATA)}  ({bands.shape[0]} bands, {len(points)} k), E_F = {e_fermi} eV")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
for band in bands:
    ax.plot(distance, band - e_fermi, color="black", lw=0.6)
models = {}
for i, (label, folder) in enumerate(WANNIER.items()):
    w90 = W90(str(folder), PREFIX)
    models[label] = w90
    model = w90.model(zero_energy=0, min_hopping_norm=1e-5, max_distance=None, ignorable_imaginary_part=None)
    evals = np.asarray(model.solve_ham(points)) - e_fermi
    for j, band in enumerate(evals.T):
        ax.plot(distance, band, color=f"C{i + 1}", lw=0.9, ls="--", label=label if j == 0 else None)
if WANNIER:
    ax.legend(loc="upper right", frameon=False)
xticks = [distance[i] for i, _ in ticks]
ax.set_xticks(xticks, [label for _, label in ticks])
for x in xticks:
    ax.axvline(x, color="gray", lw=0.4)
ax.axhline(0, color="red", lw=0.6, ls=":")
ax.set(xlim=(distance[0], distance[-1]), ylim=E_WINDOW if SCF_OUT else None,
       ylabel=r"$E - E_F$ (eV)" if SCF_OUT else "E (eV)", title=PREFIX)
plt.tight_layout()
plt.show()

In [ ]:
if SHOW_HOPPINGS and models:
    fig, axes = plt.subplots(1, len(models), figsize=(5 * len(models), 3.5), squeeze=False)
    for ax, (name, w90) in zip(axes[0], models.items()):
        distance, hopping = w90.dist_hop()
        ax.scatter(distance, np.abs(hopping), s=1)
        ax.set(yscale="log", xlabel="distance (Å)", ylabel="|H| (eV)", title=name)
    plt.tight_layout()
    plt.show()